<a href="https://colab.research.google.com/github/SAINIDHI2005/IDS_GNN_Repo/blob/main/GAT/HostFlowTemporal_GAT_training_cos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch-geometric -q

In [2]:
import os

import pandas as pd
import numpy as np

import torch

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from torch_geometric.data import Data

In [3]:
DATASET_DIR = "/Users/balji/OneDrive/GNN_IDS/Balanced_6Class_400K"

files = {
    "BENIGN.csv":0,
    "DDoS.csv":1,
    "DoS.csv":1,
    "Mirai.csv":1,
    "Recon.csv":1,
    "Spoofing.csv":1
}

dfs = []

for file,label in files.items():

    path = os.path.join(DATASET_DIR,file)

    df = pd.read_csv(path)

    attack_name = file.replace(".csv","")

    df["attack_type"] = attack_name

    df["label"] = label

    dfs.append(df)

data = pd.concat(
    dfs,
    ignore_index=True
)

print(data.shape)

print(data["attack_type"].value_counts())

(400000, 88)
attack_type
BENIGN      200000
DDoS         40000
DoS          40000
Mirai        40000
Recon        40000
Spoofing     40000
Name: count, dtype: int64


In [4]:
data = data.sort_values(
    "bidirectional_first_seen_ms"
).reset_index(drop=True)

data["flow_id"] = np.arange(len(data))

print(data.shape)

(400000, 89)


In [5]:
attack_type_backup = data["attack_type"].copy()

In [6]:
DROP_COLS = [

    "id",
    "expiration_id",

    "src_ip",
    "dst_ip",

    "src_mac",
    "dst_mac",

    "src_oui",
    "dst_oui",

    "requested_server_name",

    "client_fingerprint",
    "server_fingerprint",

    "user_agent",
    "content_type",

    "application_name",
    "application_category_name",

    "flow_id",
    "label",
    "attack_type",

    "bidirectional_first_seen_ms",
    "bidirectional_last_seen_ms",

    "src2dst_first_seen_ms",
    "src2dst_last_seen_ms",

    "dst2src_first_seen_ms",
    "dst2src_last_seen_ms",

    "application_confidence",
    "application_is_guessed",

    "vlan_id",
    "tunnel_id",

    "bidirectional_urg_packets",
    "src2dst_urg_packets",
    "dst2src_urg_packets",

    "dst2src_cwr_packets",
    "dst2src_ece_packets",

    "src_port",

    "bidirectional_duration_ms",
    "bidirectional_packets",

    "src2dst_packets",
    "dst2src_packets",

    "dst2src_min_ps",
    "dst2src_mean_ps",

    "bidirectional_min_piat_ms",
    "bidirectional_stddev_piat_ms",

    "dst2src_min_piat_ms",
    "dst2src_mean_piat_ms",
    "dst2src_max_piat_ms",

    "bidirectional_ece_packets",
    "bidirectional_psh_packets",
    "bidirectional_fin_packets",

    "src2dst_syn_packets",
    "src2dst_cwr_packets",
    "src2dst_ack_packets",
    "src2dst_rst_packets",
    "src2dst_fin_packets",

    "src2dst_mean_piat_ms",
    "src2dst_stddev_ps",

    "dst2src_ack_packets",
    "dst2src_psh_packets",
    "dst2src_rst_packets",
    "dst2src_fin_packets",

]

feature_cols = [

    c
    for c in data.columns
    if c not in DROP_COLS
]

print("Features:",len(feature_cols))

Features: 30


In [7]:
feature_cols = [
    c
    for c in data.columns
    if c not in DROP_COLS
]

print("Features:", len(feature_cols))

print("\nRetained Features:\n")
for i, f in enumerate(feature_cols, start=1):
    print(f"{i:2d}. {f}")

Features: 30

Retained Features:

 1. dst_port
 2. protocol
 3. ip_version
 4. bidirectional_bytes
 5. src2dst_duration_ms
 6. src2dst_bytes
 7. dst2src_duration_ms
 8. dst2src_bytes
 9. bidirectional_min_ps
10. bidirectional_mean_ps
11. bidirectional_stddev_ps
12. bidirectional_max_ps
13. src2dst_min_ps
14. src2dst_mean_ps
15. src2dst_max_ps
16. dst2src_stddev_ps
17. dst2src_max_ps
18. bidirectional_mean_piat_ms
19. bidirectional_max_piat_ms
20. src2dst_min_piat_ms
21. src2dst_stddev_piat_ms
22. src2dst_max_piat_ms
23. dst2src_stddev_piat_ms
24. bidirectional_syn_packets
25. bidirectional_cwr_packets
26. bidirectional_ack_packets
27. bidirectional_rst_packets
28. src2dst_ece_packets
29. src2dst_psh_packets
30. dst2src_syn_packets


In [8]:
top30 = {
    "bidirectional_mean_ps",
    "dst2src_syn_packets",
    "src2dst_min_ps",
    "ip_version",
    "src2dst_max_piat_ms",
    "dst2src_stddev_piat_ms",
    "bidirectional_max_piat_ms",
    "bidirectional_rst_packets",
    "src2dst_max_ps",
    "src2dst_duration_ms",
    "src2dst_bytes",
    "src2dst_ece_packets",
    "bidirectional_max_ps",
    "dst2src_bytes",
    "bidirectional_bytes",
    "dst2src_max_ps",
    "bidirectional_min_ps",
    "bidirectional_syn_packets",
    "dst2src_stddev_ps",
    "bidirectional_ack_packets",
    "protocol",
    "dst_port",
    "bidirectional_mean_piat_ms",
    "dst2src_duration_ms",
    "src2dst_mean_ps",
    "src2dst_min_piat_ms",
    "bidirectional_stddev_ps",
    "src2dst_stddev_piat_ms",
    "bidirectional_cwr_packets",
    "src2dst_psh_packets"
}

retained = set(feature_cols)

print("Missing from retained:")
print(sorted(top30 - retained))

print("\nExtra retained features:")
print(sorted(retained - top30))

Missing from retained:
[]

Extra retained features:
[]


In [9]:
for col in feature_cols:

    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

data[feature_cols] = (
    data[feature_cols]
    .fillna(0)
)

In [10]:
scaler = StandardScaler()

X = scaler.fit_transform(
    data[feature_cols]
)

print(X.shape)

(400000, 30)


In [11]:
all_hosts = pd.concat(
    [
        data["src_ip"],
        data["dst_ip"]
    ]
).unique()

host_to_id = {

    host : idx + len(data)

    for idx,host
    in enumerate(all_hosts)
}

num_flow_nodes = len(data)

num_host_nodes = len(all_hosts)

print("Flow Nodes:",num_flow_nodes)
print("Host Nodes:",num_host_nodes)

Flow Nodes: 400000
Host Nodes: 4572


In [12]:
edges = []

for flow_id,row in data.iterrows():

    src_host = host_to_id[
        row["src_ip"]
    ]

    dst_host = host_to_id[
        row["dst_ip"]
    ]

    edges.append(
        [flow_id,src_host]
    )

    edges.append(
        [src_host,flow_id]
    )

    edges.append(
        [flow_id,dst_host]
    )

    edges.append(
        [dst_host,flow_id]
    )

print("Host-flow edges built")

Host-flow edges built


In [13]:
src_groups = data.groupby(
    "src_ip"
)["flow_id"].apply(list)

print(
    "Groups:",
    len(src_groups)
)

Groups: 1303


In [14]:
# ==================================
# TEMPORAL FLOW-FLOW EDGES
# ==================================

src_groups = data.groupby("src_ip")["flow_id"].apply(list)

temporal_edges = 0

for flow_list in src_groups:

    n = len(flow_list)

    if n < 2:
        continue

    for i in range(n):

        f1 = flow_list[i]

        for k in [1, 2, 3]:

            if i + k < n:

                f2 = flow_list[i + k]

                edges.append([f1, f2])
                edges.append([f2, f1])

                temporal_edges += 2

print("Temporal edges added:", temporal_edges)
print("Total edges:", len(edges))

Temporal edges added: 2388190
Total edges: 3988190


In [15]:
edge_index = torch.tensor(
    edges,
    dtype=torch.long
).t().contiguous()

print(edge_index.shape)

torch.Size([2, 3988190])


In [16]:
flow_features = torch.tensor(
    X,
    dtype=torch.float
)

In [17]:
host_features = torch.zeros(

    (
        num_host_nodes,
        flow_features.shape[1]
    ),

    dtype=torch.float
)

host_counts = np.zeros(
    num_host_nodes
)

for flow_id,row in data.iterrows():

    src_idx = (
        host_to_id[
            row["src_ip"]
        ]
        -
        num_flow_nodes
    )

    dst_idx = (
        host_to_id[
            row["dst_ip"]
        ]
        -
        num_flow_nodes
    )

    host_features[src_idx] += flow_features[flow_id]
    host_features[dst_idx] += flow_features[flow_id]

    host_counts[src_idx] += 1
    host_counts[dst_idx] += 1

for i in range(num_host_nodes):

    if host_counts[i] > 0:

        host_features[i] /= (
            host_counts[i]
        )

In [18]:
x = torch.cat(
    [
        flow_features,
        host_features
    ],
    dim=0
)

print(x.shape)

torch.Size([404572, 30])


In [19]:
flow_labels = torch.tensor(
    data["label"].values,
    dtype=torch.long
)

host_labels = torch.full(
    (
        num_host_nodes,
    ),
    -1,
    dtype=torch.long
)

y = torch.cat(
    [
        flow_labels,
        host_labels
    ]
)

print(y.shape)

torch.Size([404572])


In [20]:
flow_hosts = (
    data["src_ip"].astype(str)
    + "_"
    + data["dst_ip"].astype(str)
)

unique_pairs = flow_hosts.unique()

train_pairs, val_pairs = train_test_split(
    unique_pairs,
    test_size=0.2,
    random_state=42
)

train_idx = data[
    flow_hosts.isin(train_pairs)
].index.values

val_idx = data[
    flow_hosts.isin(val_pairs)
].index.values

print("Train flows:", len(train_idx))
print("Validation flows:", len(val_idx))

print("Train pairs:", len(train_pairs))
print("Validation pairs:", len(val_pairs))

Train flows: 313332
Validation flows: 86668
Train pairs: 9756
Validation pairs: 2439


In [21]:
total_nodes = (
    num_flow_nodes
    +
    num_host_nodes
)

train_mask = torch.zeros(total_nodes,dtype=torch.bool)

val_mask = torch.zeros(total_nodes,dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True

print("Train mask:", train_mask.sum().item())
print("Validation mask :",val_mask.sum().item())

Train mask: 313332
Validation mask : 86668


C:\Users\balji\AppData\Local\Temp\ipykernel_29900\3493698591.py:11: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  train_mask[train_idx] = True


In [22]:
graph = Data(
    x=x,
    edge_index=edge_index,
    y=y,
    train_mask=train_mask,
    val_mask=val_mask
)
print(graph)

print(
    "Nodes:",
    graph.num_nodes
)

print(
    "Edges:",
    graph.num_edges
)

Data(x=[404572, 30], edge_index=[2, 3988190], y=[404572], train_mask=[404572], val_mask=[404572])
Nodes: 404572
Edges: 3988190


In [23]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv

class GATModel(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        num_classes,
        heads=4
    ):
        super().__init__()

        # Layer 1
        self.conv1 = GATv2Conv(
            in_channels,
            hidden_channels // heads,
            heads=heads
        )

        # Layer 2
        self.conv2 = GATv2Conv(
            hidden_channels,
            hidden_channels // heads,
            heads=heads
        )

        # Layer 3: Output matches hidden_channels // 2 (128)
        self.conv3 = GATv2Conv(
            hidden_channels,
            (hidden_channels // 2) // heads,
            heads=heads
        )

        self.bn1 = torch.nn.BatchNorm1d(hidden_channels)
        self.bn2 = torch.nn.BatchNorm1d(hidden_channels)
        self.bn3 = torch.nn.BatchNorm1d(hidden_channels // 2)

        self.classifier = torch.nn.Linear(
            hidden_channels // 2,
            num_classes
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.4, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.4, training=self.training)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        x = self.classifier(x)

        return x

In [24]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [25]:
model = GATModel(
    in_channels=graph.num_node_features,
    hidden_channels=256,
    num_classes=2
)

model = model.to(device)

graph = graph.to(device)

print(model)

GATModel(
  (conv1): GATv2Conv(30, 64, heads=4)
  (conv2): GATv2Conv(256, 64, heads=4)
  (conv3): GATv2Conv(256, 32, heads=4)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)


In [26]:
from torch_geometric.loader import NeighborLoader

train_loader = NeighborLoader(
    graph,
    input_nodes=graph.train_mask,
    num_neighbors=[15, 10],
    batch_size=2048,
    shuffle=True
)

val_loader = NeighborLoader(
    graph,
    input_nodes=graph.val_mask,
    num_neighbors=[15, 10],
    batch_size=2048,
    shuffle=False
)

print("Train batches :", len(train_loader))
print("Validation batches :", len(val_loader))

Train batches : 153
Validation batches : 43


In [27]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.002,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=150,
    eta_min=1e-5
)

In [28]:
weights = torch.tensor(
    [1.0, 1.5],
    dtype=torch.float
).to(device)

criterion = torch.nn.CrossEntropyLoss(
    weight=weights
)

In [29]:
def train():

    model.train()

    total_loss = 0

    for batch in train_loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        out = model(
            batch.x,
            batch.edge_index
        )

        loss = criterion(
            out[:batch.batch_size],
            batch.y[:batch.batch_size]
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [30]:
@torch.no_grad()

def evaluate(loader):

    model.eval()

    correct = 0
    total = 0

    for batch in loader:

        batch = batch.to(device)

        out = model(
            batch.x,
            batch.edge_index
        )

        pred = out[:batch.batch_size].argmax(dim=1)

        correct += (
            pred ==
            batch.y[:batch.batch_size]
        ).sum().item()

        total += batch.batch_size

    return correct / total

In [31]:
import os
import time
import joblib
import torch

best_val_acc = 0.0
best_epoch = 0

SAVE_DIR = r"/Users/balji/OneDrive/GNN_IDS/HostFlow_Saved_Model"
os.makedirs(SAVE_DIR, exist_ok=True)

start_time = time.time()

for epoch in range(51, 151):

   loss = train()

   train_acc = evaluate(train_loader)

   val_acc = evaluate(val_loader)

   # Save best model based on validation accuracy
   if val_acc > best_val_acc:

        best_val_acc = val_acc
        best_epoch = epoch

        torch.save(
            model.state_dict(),
            os.path.join(
                SAVE_DIR,
                "HostFlowTemporal_GATmodel.pth"
            )
        )

        joblib.dump(
            scaler,
            os.path.join(
                SAVE_DIR,
                "HostFlowTemporal _SCALERmodel.pkl"
            )
        )
   scheduler.step()

   print(
        f"Epoch {epoch:03d}"
        f"LR: {optimizer.param_groups[0]['lr']:.6f}"
        f" | Loss {loss:.4f}"
        f" | Train {train_acc:.4f}"
        f" | Validation {val_acc:.4f}"

   )

end_time = time.time()

training_time = end_time - start_time

print("\n" + "=" * 60)
print(f"Training Time          : {training_time:.2f} seconds")
print(f"Best Epoch             : {best_epoch}")
print(f"Best Validation Acc.   : {best_val_acc:.4f}")
print(f"Model Saved At         : {os.path.join(SAVE_DIR,'HostFlowTemporal_GATmodel.pth')}")
print(f"Scaler Saved At        : {os.path.join(SAVE_DIR,'HostFlowTemporal _SCALERmodel.pkl')}")
print("=" * 60)
print("Training Complete.")

Epoch 001LR: 0.002000 | Loss 0.2289 | Train 0.9480 | Validation 0.9232
Epoch 002LR: 0.001999 | Loss 0.1277 | Train 0.9700 | Validation 0.9410
Epoch 003LR: 0.001998 | Loss 0.0987 | Train 0.9697 | Validation 0.9481
Epoch 004LR: 0.001997 | Loss 0.0920 | Train 0.9752 | Validation 0.9562
Epoch 005LR: 0.001995 | Loss 0.0762 | Train 0.9804 | Validation 0.9680
Epoch 006LR: 0.001992 | Loss 0.0693 | Train 0.9830 | Validation 0.9670
Epoch 007LR: 0.001989 | Loss 0.0600 | Train 0.9822 | Validation 0.9655
Epoch 008LR: 0.001986 | Loss 0.0571 | Train 0.9814 | Validation 0.9573
Epoch 009LR: 0.001982 | Loss 0.0529 | Train 0.9857 | Validation 0.9713
Epoch 010LR: 0.001978 | Loss 0.0500 | Train 0.9846 | Validation 0.9657
Epoch 011LR: 0.001974 | Loss 0.0469 | Train 0.9894 | Validation 0.9723
Epoch 012LR: 0.001969 | Loss 0.0432 | Train 0.9877 | Validation 0.9693
Epoch 013LR: 0.001963 | Loss 0.0399 | Train 0.9883 | Validation 0.9657
Epoch 014LR: 0.001958 | Loss 0.0407 | Train 0.9898 | Validation 0.9733
Epoch 

In [32]:
import time
import torch

print("=" * 60)

# Training Time
print(f"Training Time: {training_time:.2f} seconds ({150} epochs)\n")

# GPU Memory
if torch.cuda.is_available():

    allocated = torch.cuda.memory_allocated() / (1024 ** 2)
    reserved = torch.cuda.memory_reserved() / (1024 ** 2)
    max_reserved = torch.cuda.max_memory_reserved() / (1024 ** 2)

    print(f"Allocated GPU Memory: {allocated:.2f} MB")
    print(f"Reserved GPU Memory: {reserved:.2f} MB")
    print(f"Max GPU Memory: {max_reserved:.2f} MB\n")

# Model Parameters
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters: {total_params}")
print(f"Trainable Parameters: {trainable_params}\n")

# Graph Memory
node_memory = (
    graph.x.element_size()
    * graph.x.nelement()
) / (1024 ** 2)

edge_memory = (
    graph.edge_index.element_size()
    * graph.edge_index.nelement()
) / (1024 ** 2)

graph_memory = node_memory + edge_memory

print(f"Node Feature Memory: {node_memory:.2f} MB")
print(f"Edge Memory: {edge_memory:.2f} MB")
print(f"Total Graph Memory: {graph_memory:.2f} MB")

print("=" * 60)

Training Time: 850.12 seconds (150 epochs)

Allocated GPU Memory: 131.57 MB
Reserved GPU Memory: 3164.00 MB
Max GPU Memory: 3164.00 MB

Total Parameters: 216066
Trainable Parameters: 216066

Node Feature Memory: 46.30 MB
Edge Memory: 60.85 MB
Total Graph Memory: 107.15 MB
